# Finetuned mE5 Experiment Setup & Import

This notebook helps you prepare the `results/experiments.csv` for the finetuned mE5 model (`finetuned-me5`). 

**Goals:**
1. Generate the expected IDs for these experiments.
2. Seed the CSV with placeholder rows if you have results on the server but not yet locally.
3. Provide a way to manually sync server metrics into your local tracker.

## 1. Environment Check
Verify if the local embedding file is present. If it is missing, `run_finetuned_experiments.py` will not be able to compute metrics locally.

In [1]:
import os
embedding_path = 'results/embeddings/finetuned-me5_embeddings.csv'
if os.path.exists(embedding_path):
    print(f"✅ SUCCESS: {embedding_path} found locally.")
else:
    print(f"❌ MISSING: {embedding_path} not found. \nYou should download this from the server if you want to run the experiments locally.")

❌ MISSING: results/embeddings/finetuned-me5_embeddings.csv not found. 
You should download this from the server if you want to run the experiments locally.


## 2. Seed Missing Experiment Records
This code identifies which experiments are needed for `finetuned-me5` and adds them as "Pending" rows to the tracker.

In [2]:
import pandas as pd
from orgpackage.aux import load_experiments, get_id

df = load_experiments('results/experiments.csv')
MODEL_NAME = 'finetuned-me5'
DOMAINS = ['medical', 'administrative', 'education']
METHODS = ['similarity', 'classifier']
CLASSIFIERS = ['logreg', 'svm']
added_count = 0

def experiment_exists(domain, method, model, clf=None):
    matching = df[(df['Domain'] == domain) & (df['Method'] == method)]
    for _, row in matching.iterrows():
        params = row['Parameters']
        if params.get('model') == model:
            if method == 'similarity' or params.get('classifier') == clf:
                return True
    return False

new_rows = []
for domain in DOMAINS:
    # 1. Similarity
    if not experiment_exists(domain, 'similarity', MODEL_NAME):
        eid = get_id(df, domain, 'embedding', 'similarity')
        params = {'structure': '3-multiclass', 'model': MODEL_NAME, 'distance': 0.1, 'n_shot': '0_shot'}
        new_rows.append([eid, domain, 'embedding', 'similarity', params, None, None, None])
        print(f"Created seed for {eid}")
    
    # 2. Classifier
    for clf in CLASSIFIERS:
        if not experiment_exists(domain, 'classifier', MODEL_NAME, clf):
            eid = get_id(df, domain, 'embedding', 'classifier')
            params = {'structure': '3-multiclass', 'model': MODEL_NAME, 'classifier': clf}
            new_rows.append([eid, domain, 'embedding', 'classifier', params, None, None, None])
            print(f"Created seed for {eid}")

if new_rows:
    new_df = pd.DataFrame(new_rows, columns=df.columns)
    df = pd.concat([df, new_df], ignore_index=True)
    # We do not save to CSV automatically to allow verification
    print(f"\nPrepared {len(new_rows)} pending records. Run the next cell to save.")
else:
    print("All finetuned-me5 records already exist in the dataframe.")

Generating experiment med-e-similarity-12
Created seed for med-e-similarity-12
Generating experiment med-e-classifier-8
Created seed for med-e-classifier-8
Generating experiment med-e-classifier-8
Created seed for med-e-classifier-8
Generating experiment adm-e-similarity-12
Created seed for adm-e-similarity-12
Generating experiment adm-e-classifier-8
Created seed for adm-e-classifier-8
Generating experiment adm-e-classifier-8
Created seed for adm-e-classifier-8
Generating experiment edu-e-similarity-12
Created seed for edu-e-similarity-12
Generating experiment edu-e-classifier-8
Created seed for edu-e-classifier-8
Generating experiment edu-e-classifier-8
Created seed for edu-e-classifier-8

Prepared 9 pending records. Run the next cell to save.


## 3. Save to experiments.csv
Only run this if the preview above looks correct.

In [ ]:
import json
# Ensure Parameters are stringified for CSV
df_to_save = df.copy()
df_to_save['Parameters'] = df_to_save['Parameters'].apply(
    lambda x: json.dumps(x) if isinstance(x, dict) else x
)
df_to_save.to_csv('results/experiments.csv', index=False)
print("results/experiments.csv updated with pending records.")

## 4. Manual Sync from Server
If you have the metrics from the server, you can fill them in here using the IDs generated above.

In [ ]:
# Example dictionary of server results
# server_results = {
#    'med-e-similarity-6': {'Accuracy': 0.85, 'Recall': 0.82, 'F1': 0.835},
# }

server_results = {
    # FILL ME IN WITH SERVER DATA
}

if server_results:
    df = load_experiments('results/experiments.csv')
    for eid, metrics in server_results.items():
        mask = df['ID'] == eid
        if mask.any():
            for k, v in metrics.items():
                df.loc[mask, k] = v
            print(f"Updated {eid} with server data.")
    
    # Save final
    df_to_save = df.copy()
    df_to_save['Parameters'] = df_to_save['Parameters'].apply(
        lambda x: json.dumps(x) if isinstance(x, dict) else x
    )
    df_to_save.to_csv('results/experiments.csv', index=False)
    print("Sync complete.")